# Stage 7: Modern Packaging, CI/CD, and Lightweight Test Fixtures

This notebook demonstrates:
1. Validating package metadata and dependencies via `pyproject.toml`
2. Fast unit testing in CI environments using synthetic dummy models
3. Running TFLite inference with zero external weight downloads
4. Inspecting GitHub Actions CI workflows for automated pull-request validation

All configurations are centralized in `pyproject.toml`, `.github/workflows/ci.yml`, and `tests/conftest.py`.

In [ ]:
import sys
from pathlib import Path
import tomllib
import yaml
import numpy as np

# Add src to Python path
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from catdog import __version__
from catdog.config import Config
from catdog.predict import CatDogPredictor

print(f"CatDog version: {__version__}")

## 1. Inspecting Packaging Configuration (pyproject.toml)

In [ ]:
with open(project_root / "pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

print("Project Name:", pyproject["project"]["name"])
print("Version:", pyproject["project"]["version"])
print("CLI Scripts:", pyproject["project"]["scripts"])
print("Optional Groups:", list(pyproject["project"]["optional-dependencies"].keys()))

## 2. Fast CI Dummy Model Generation

In continuous integration (CI) runners, downloading or training a 25MB+ neural network for every commit is slow, costly, and brittle. Here we synthesize a minimal architecture (~1K parameters) that matches the production signature.

In [ ]:
import tensorflow as tf

def create_dummy_model():
    inp = tf.keras.layers.Input(shape=(150, 150, 3), name="image_input")
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_rescaling")(inp)
    x = tf.keras.layers.Conv2D(4, (3, 3), activation="relu", name="conv_tiny")(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(1, activation="sigmoid", name="predictions")(x)
    model = tf.keras.Model(inputs=inp, outputs=x, name="tiny_ci_model")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

tiny = create_dummy_model()
print(f"Dummy model parameter count: {tiny.count_params():,}")
tiny_path = project_root / "models" / "ci_dummy_model.keras"
tiny.save(tiny_path)
print(f"Saved dummy model to: {tiny_path} ({tiny_path.stat().st_size / 1024:.1f} KB)")

## 3. End-to-End Inference Verification with Dummy Model

In [ ]:
from PIL import Image

# Create sample in-memory image
sample_img_path = project_root / "reports" / "ci_sample.jpg"
img = Image.new("RGB", (150, 150), color=(120, 180, 240))
img.save(sample_img_path)

predictor = CatDogPredictor(model_path=tiny_path)
result = predictor.predict_image(sample_img_path)

print("Prediction Output:")
for k, v in result.items():
    print(f"  {k:16s}: {v}")

# Clean up temporary artifacts
sample_img_path.unlink(missing_ok=True)
tiny_path.unlink(missing_ok=True)

## 4. Inspecting GitHub Actions CI Workflow

In [ ]:
ci_path = project_root / ".github" / "workflows" / "ci.yml"
with open(ci_path, "r", encoding="utf-8") as f:
    ci_content = f.read()

print(ci_content[:450] + "\n... [remaining workflow definition truncated]")